In [1]:
import os
import sys
from google.colab import drive

# Mount Drive and set Project Path
drive.mount('/content/drive', force_remount=True)
PROJECT_PATH = '/content/drive/MyDrive/CILP_Project'
sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

# Unzip Data to local runtime
if not os.path.exists('/content/data/assessment'):
    print(" Extracting multimodal data...")
    !unzip -q assessment.zip -d /content/data/
    print(" Extraction complete.")

Mounted at /content/drive
 Extracting multimodal data...
 Extraction complete.


In [2]:
!pip install -r "{PROJECT_PATH}/requirements_colab.txt"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.8/112.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.3/316.3 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.5/934.5 kB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 19.5 MB/s eta 

In [3]:
import torch
import torch.nn as nn
import wandb
from torch.utils.data import DataLoader
from src.datasets import MultimodalDataset
from src.models import MultiTaskModel
from src.training import train_one_epoch
from src.utils import set_seeds

#Configuration for Reproducibility
set_seeds(51)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
DATA_ROOT = '/content/data/assessment'
CHECKPOINT_DIR = '/content/drive/MyDrive/CILP_Project/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Data Loaders (Using 10% subset for rapid experimentation)

train_loader = DataLoader(MultimodalDataset(DATA_ROOT, subset=0.1), batch_size=32, shuffle=True)
val_loader = DataLoader(MultimodalDataset(DATA_ROOT, subset=0.02), batch_size=32)

print(f" Setup Complete. Training on {len(train_loader.dataset)} samples.")

 Deterministic environment initialized (Seed: 51)
Scanning dataset at /content/data/assessment...
Loaded 10751 total samples.
 Subset active: Keeping random 1075 samples.
Scanning dataset at /content/data/assessment...
Loaded 10751 total samples.
 Subset active: Keeping random 215 samples.
 Setup Complete. Training on 1075 samples.


In [4]:
import time
import pandas as pd
import wandb
import torch
import torch.nn.functional as F
import os
from IPython.display import display

# Task 4 Configuration
ablation_experiments = [
    {"use_strided": False, "name": "Task4_Baseline_MaxPool"},
    {"use_strided": True,  "name": "Task4_Ablation_Strided"}
]

final_results = []
BATCH_SIZE = 32
LR = 1e-3
EPOCHS = 50

for cfg in ablation_experiments:
    print(f"\n Starting Experiment: {cfg['name']}")


    if wandb.run is not None: wandb.finish()

    # FIX 2: Hyperparameter Configuratio
    # We log all config details at initialization
    config = {
        "architecture": "Strided_Conv" if cfg['use_strided'] else "MaxPool2d",
        "learning_rate": LR,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "optimizer": "Adam",
        "embedding_size": 128, # Final layer size
        "fusion_strategy": "intermediate_concat", # Same as Task 3
        "task": "Task 4 Ablation"
    }

    wandb.init(
        project="cilp-extended-assessment",
        name=cfg['name'],
        config=config,
        reinit=True
    )

    # Initialize Model
    model = MultiTaskModel(
        mode='intermediate',
        strategy='concat',
        use_strided=cfg['use_strided']
    ).to(device)

    # Log Parameter Count to Config
    param_count = sum(p.numel() for p in model.parameters())
    wandb.config.update({"num_parameters": param_count})
    print(f"   Parameters: {param_count:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    total_time = 0
    best_acc = 0

    for epoch in range(EPOCHS):
        start_time = time.time()

        # Train
        train_loss = train_one_epoch(model, train_loader, optimizer, device, is_cilp=False)

        epoch_duration = time.time() - start_time
        total_time += epoch_duration

        # Validation
        model.eval()
        correct, total = 0, 0
        val_loss = 0.0 # Track Validation Loss

        # Storage for Sample Predictions
        sample_images = []

        with torch.no_grad():
            for i, (rgb, lidar, labels) in enumerate(val_loader):
                rgb, lidar, labels = rgb.to(device), lidar.to(device), labels.to(device)
                outputs = model(rgb, lidar)

                # Shape fix
                if outputs.shape != labels.shape: outputs = outputs.view(-1)

                # Calculate Val Loss
                batch_loss = F.binary_cross_entropy_with_logits(outputs, labels.float())
                val_loss += batch_loss.item()

                preds = (torch.sigmoid(outputs) > 0.5).float()
                correct += (preds == labels).sum().item()
                total += labels.size(0)


                if epoch == EPOCHS - 1 and len(sample_images) < 5:
                    for j in range(min(5 - len(sample_images), len(rgb))):
                        # Convert tensor to image for W&B
                        img = rgb[j].permute(1, 2, 0).cpu().numpy()
                        pred_lbl = "Sphere" if preds[j].item() == 1 else "Cube"
                        true_lbl = "Sphere" if labels[j].item() == 1 else "Cube"

                        sample_images.append(
                            wandb.Image(img, caption=f"Pred: {pred_lbl} | GT: {true_lbl}")
                        )

        val_acc = correct / total
        val_loss /= len(val_loader)

        #Log All Metrics
        log_dict = {
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "epoch_time": epoch_duration,
            "learning_rate": LR,
            "epoch": epoch
        }

        # Log samples only on last epoch
        if sample_images:
            log_dict["sample_predictions"] = sample_images

        wandb.log(log_dict)

        print(f"   Ep {epoch+1}: T-Loss {train_loss:.4f} | V-Loss {val_loss:.4f} | Acc {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            if not os.path.exists(CHECKPOINT_DIR): os.makedirs(CHECKPOINT_DIR)
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f"{cfg['name']}.pth"))

    # Table Data
    final_results.append({
        "Architecture": "Strided Conv" if cfg['use_strided'] else "MaxPool2d",
        "Val Loss": val_loss,
        "Parameters": param_count,
        "Avg Time/Epoch": f"{total_time/EPOCHS:.2f}s",
        "Final Accuracy": best_acc
    })

    wandb.finish()

print("\n--- Task 4 Comparison Table ---")
df = pd.DataFrame(final_results)
display(df)


 Starting Experiment: Task4_Baseline_MaxPool


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


...

   Ep 49: T-Loss 0.0002 | V-Loss 0.0001 | Acc 1.0000


   Ep 50: T-Loss 0.0002 | V-Loss 0.0001 | Acc 1.0000


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
epoch_time,█▂▁▁▁▅▅▁▁▂▁▃▅▄▂▁▁▂▄▅▁▁▁▁▂▄▁▁▁▁▃▅▃▁▁▁▁▄▅▁
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▇▇▆▆▆▆▆▆▅▅▄▄▄▃▃▂▃▃▂▂▂▂▂▁▁▃▂▁▂▁▁▁▁▁▁▁▁▁▁
val_acc,▂▂▂▂▂▂▂▂▂▂▂▂▁▂▅▆▆▇▄▇█▅█▇▆██▅▃█████▇█████
val_loss,▇█▆▇▆▆▆▆▆▆▅▄▆▄▃▃▃▂▃▂▁▃▁▂▂▁▁▃▃▁▁▁▁▁▁▁▁▁▁▁
epoch,49
epoch_time,1.57837
learning_rate,0.001
train_loss,0.00016
val_acc,1



--- Task 4 Comparison Table ---


,Architecture,Val Loss,Parameters,Avg Time/Epoch,Final Accuracy
0,MaxPool2d,0.000011,372743,1.25s,1.0
1,Strided Conv,0.000057,598043,1.64s,1.0
